# Modelo Make-or-Buy Multiestágio

## 1. Introdução

* **Objetivo:** minimizar o custo para atender a demanda de itens, combinando estoque, produção interna e compra externa.
* **Modelagem em grafo:** itens são nós; receitas são arcos ponderados que consomem insumos para gerar um produto.
* **Receitas alternativas:** um mesmo item pode ter mais de uma receita concorrendo no solver (ex.: Computador pode ser produzido de 3 formas diferentes).

### Aplicações reais
* **Petroquímica:** refino e craqueamento com recirculação de frações.
* **Farmacêutica:** síntese em batelada com reaproveitamento de reagentes.
* **Engenharia de processos:** planejamento multiestágio de plantas industriais.

## 2. Modelo Matemático

Programação Linear sobre $V$ (itens) e $R$ (receitas), cada receita $r \in R$ produz um único item $p(r) \in V$ e consome insumos $(i, a_{i,r})$.

### Variáveis
* $x_i \ge 0$: quantidade comprada do item $i$.
* $s_i \ge 0$: estoque inicial utilizado, $s_i \le E_i$.
* $z_r \ge 0$: quantidade produzida pela receita $r$.

### Objetivo

$$\min Z = \sum_{i \in V} C^{\text{compra}}_i \cdot x_i + \sum_{r \in R} C^{\text{prod}}_r \cdot z_r$$

### Restrições

$$x_i + s_i + \sum_{r:\, p(r)=i} z_r \;-\; \sum_{r \in R} a_{i,r} \cdot z_r \;\ge\; D_i, \quad \forall i \in V$$

$$s_i \le E_i, \qquad x_i, s_i, z_r \ge 0$$

* **Viabilidade garantida:** $x_i$ sem limite superior → sempre existe solução.
* **Otimalidade finita:** todos os custos são estritamente positivos → nunca compensa produzir/comprar além do necessário.

## 3. Fonte de Dados

Custos extraídos do arquivo oficial do jogo **Docs.json** (Satisfactory), em vez de valores fictícios:

* **Custo de compra** ($C^{\text{compra}}_i$): pontos do **AWESOME Sink** (`mResourceSinkPoints`) de cada item.
* **Custo de produção** ($C^{\text{prod}}_r$): potência da máquina (`mPowerConsumption`) × duração da receita (`mManufactoringDuration`), por unidade produzida.
* Minérios brutos (`FGResourceDescriptor`) e itens fabricados (`FGItemDescriptor`) são tratados igualmente como nós do grafo.

## 4. Solução

### Pipeline

`Docs.json` → padronização (itens/receitas) → validação → solver (`scipy.optimize.linprog`) → resultados por cenário.

* 15 cenários (ordens 10 / 100 / 1000 itens) × 6 métodos de solver × 10 repetições = 900 execuções.

### Desempenho por solver

* Métodos modernos (`highs`, `highs-ds`, `highs-ipm`, `revised simplex`) convergem ao mesmo ótimo em 100% dos cenários.
* Métodos legados (`simplex`, `interior-point`) falham em instâncias maiores.

<img src="results/satisfactory/graficos/tempo_por_solver.png" width="650"/>
<img src="results/satisfactory/graficos/memoria_por_solver.png" width="650"/>

### Escalabilidade do custo ótimo

O custo total cresce com a ordem do cenário (escala log), refletindo cadeias de produção mais profundas e demandas maiores.

<img src="results/satisfactory/graficos/objetivo_por_ordem.png" width="650"/>

### Make-or-buy por item

Itens de alto valor agregado (ex.: `Temporal Processor`, peças do Elevador Espacial) concentram o maior custo total; fluidos como combustível líquido são quase sempre produzidos, nunca comprados.

<img src="results/satisfactory/graficos/custo_por_item_top.png" width="650"/>
<img src="results/satisfactory/graficos/compra_vs_producao.png" width="650"/>

## 5. Discussão

* **Trade-off real:** com custos do jogo, comprar nem sempre é pior — depende da cadeia de receitas disponíveis (ex.: Computador: comprar custa mais que a melhor receita de produção).
* **Receitas alternativas importam:** permitir múltiplas receitas por item revela decisões de make-or-buy que um modelo de receita única esconderia.
* **Robustez do solver:** métodos de ponto interior/simplex revisado (`highs*`) escalam bem; métodos legados do SciPy não são confiáveis acima de instâncias pequenas.
* **Limitações:** subprodutos de receitas (ex. resíduo de óleo pesado) são ignorados; custo de produção e custo de compra usam unidades diferentes (energia vs. pontos de sink), é uma aproximação econômica, não um preço único consistente.

## 6. Referências

* [Satisfactory Wiki — AWESOME Sink](https://satisfactory.wiki.gg/wiki/AWESOME_Sink)
* [Satisfactory Wiki — Items](https://satisfactory.wiki.gg/wiki/Items)
* [Satisfactory Wiki — Recipes](https://satisfactory.wiki.gg/wiki/Recipes)
* `Docs.json` — dados oficiais do jogo, extraídos via SatisfactoryTools/CommunityResources.
* [SciPy — `scipy.optimize.linprog`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.linprog.html)